In [2]:
import pandas as pd

df = pd.read_csv('dataset.csv')


df['Date'] = pd.to_datetime(df['Date'], errors='coerce')


df['date'] = df['Date'].dt.strftime("%Y-%m-%d")
df['time'] = df['Date'].dt.strftime("%H:%M:%S")
df.info

C:\Users\Administrator\AppData\Local\Temp\ipykernel_20528\1421013379.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'], errors='coerce')


<bound method DataFrame.info of               ID Case Number       Date                     Block  IUCR  \
0       13782022    JJ188466 2025-03-20           101XX S WOOD ST  0486   
1       13784448    JJ192378 2025-03-20        001XX E PEARSON ST  1320   
2       13781020    JJ188344 2025-03-20        053XX N MOBILE AVE  0910   
3       13781612    JJ188788 2025-03-20        040XX N CICERO AVE  0860   
4       13781748    JJ189227 2025-03-20  084XX S STONY ISLAND AVE  0910   
...          ...         ...        ...                       ...   ...   
467073  13299254    JG530673 2023-01-01          036XX N BROADWAY  0620   
467074  13222722    JG438814 2023-01-01       052XX N SHERIDAN RD  0810   
467075  13297756    JG528042 2023-01-01  041XX W BELLE PLAINE AVE  1752   
467076  13291252    JG520635 2023-01-01        003XX N KEDZIE AVE  1585   
467077  13140855    JG341458 2023-01-01      082XX S JEFFERY BLVD  1754   

                      Primary Type  \
0                          BA

In [3]:
import numpy as np
df = df[df.Latitude.notnull()]

df.info


df['date'] = pd.to_datetime(df['date'], errors = 'coerce')
start_time = df['date'].min()
end_time = df['date'].max()
print(f"Start time: {start_time} \nEnd time: {end_time}")

df['Longitude'] = df['Longitude'].round(5)
df['Latitude'] = df['Latitude'].round(5)

df['weekday'] = df['date'].dt.weekday

df['hour'] = df['date'].dt.hour
df['sin_hour'] = np.sin(2 * np.pi * df['hour'] / 24).round(4)
df['cos_hour'] = np.cos(2 * np.pi * df['hour'] / 24).round(4)


df['sin_weekday'] = np.sin(2 * np.pi * df['weekday'] / 7).round(4)
df['cos_weekday'] = np.cos(2 * np.pi * df['weekday'] / 7).round(4)

df['label'] = 1

df[['cos_hour', 'sin_hour', 'cos_weekday', 'sin_weekday', 'Longitude', 'Latitude', 'label']].head()

df[['date', 'cos_hour', 'sin_hour', 'cos_weekday', 'sin_weekday', 'Longitude', 'Latitude', 'label']].to_csv('df.csv', index=False)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_20528\2584636241.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['date'] = pd.to_datetime(df['date'], errors = 'coerce')
C:\Users\Administrator\AppData\Local\Temp\ipykernel_20528\2584636241.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Longitude'] = df['Longitude'].round(5)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_20528\2584636241.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.

Start time: 2023-01-01 00:00:00 
End time: 2025-03-20 00:00:00


In [4]:
ds = pd.read_csv('df.csv')

max_lat = ds['Latitude'].max()
min_lat = ds['Latitude'].min()
max_lon = ds['Longitude'].max()
min_lon = ds['Longitude'].min()

print(f"Max Latitude: {max_lat} \nMin Latitude: {min_lat} \nMax Longitude: {max_lon} \nMin Longitude: {min_lon} \n")



Max Latitude: 42.02255 
Min Latitude: 41.64459 
Max Longitude: -87.52453 
Min Longitude: -87.93973 



In [9]:
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm import tqdm

# Load your positive dataset
ds = pd.read_csv("df.csv")
ds['date'] = pd.to_datetime(ds['date'])

# Convert key columns to numpy arrays
lats = ds['Latitude'].values
lons = ds['Longitude'].values
times = ds['date'].values.astype('datetime64[s]')

# Get range bounds
lat_min, lat_max = lats.min(), lats.max()
lon_min, lon_max = lons.min(), lons.max()
time_min, time_max = times.min(), times.max()
time_range_seconds = (time_max - time_min).astype('timedelta64[s]').astype(int)

# Target number of negative samples
target_samples = len(ds)
batch_size = 10000
negative_samples = []

# Start generating
print("Generating negative samples...")
pbar = tqdm(total=target_samples)
while len(negative_samples) < target_samples:
    rand_secs = np.random.randint(0, time_range_seconds, batch_size)
    rand_times = time_min + rand_secs.astype('timedelta64[s]')
    rand_lats = np.random.uniform(lat_min, lat_max, batch_size)
    rand_lons = np.random.uniform(lon_min, lon_max, batch_size)

    for i in range(batch_size):
        lat_diff = np.abs(lats - rand_lats[i])
        lon_diff = np.abs(lons - rand_lons[i])
        time_diff = np.abs((times - rand_times[i]).astype('timedelta64[s]').astype(int))

        # Filter out any samples too close to real points
        if not np.any((lat_diff < 0.001) & (lon_diff < 0.001) & (time_diff < 900)):
            dt = rand_times[i].astype(object)
            weekday = dt.weekday()
            hour = dt.hour

            sin_hour = np.sin(2 * np.pi * hour / 24)
            cos_hour = np.cos(2 * np.pi * hour / 24)
            sin_weekday = np.sin(2 * np.pi * weekday / 7)
            cos_weekday = np.cos(2 * np.pi * weekday / 7)

            negative_samples.append({
                'date': dt.date(),
                'cos_hour': round(cos_hour, 4),
                'sin_hour': round(sin_hour, 4),
                'cos_weekday': round(cos_weekday, 4),
                'sin_weekday': round(sin_weekday, 4),
                'Longitude': round(rand_lons[i], 5),
                'Latitude': round(rand_lats[i], 5),
                'label': 0
            })

            pbar.update(1)
            if len(negative_samples) >= target_samples:
                break

pbar.close()

# Save results
df_neg = pd.DataFrame(negative_samples)
df_neg.to_csv('neg_vals.csv', index=False)
print("Negative samples saved to neg_vals.csv")


Generating negative samples...


100%|██████████| 466131/466131 [1:11:56<00:00, 107.98it/s]


Negative samples saved to neg_vals.csv


In [ ]:
import pandas as pd

# Read the CSV files
df1 = pd.read_csv('df.csv')
df2 = pd.read_csv('neg_vals.csv')


df1_sample = df1
df2_sample = df2

# Concatenate and shuffle
combined = pd.concat([df1_sample, df2_sample]).sample(frac=1).reset_index(drop=True)

# Save to new CSV
combined.to_csv('shuffled.csv', index=False)


In [14]:
from sklearn.model_selection import train_test_split
df_combined = pd.read_csv('shuffled.csv')
X = df_combined[['Latitude', 'Longitude', 'sin_hour', 'cos_hour', 'sin_weekday', 'cos_weekday']]
y = df_combined['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [15]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

In [19]:
from sklearn.metrics import classification_report, confusion_matrix
probabilities = model.predict_proba(X_test)

# For binary classification, probabilities[:, 1] will give the probability of class 1
class_1_probabilities = probabilities[:, 1]
y_pred = model.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


[[91721  1506]
 [   95 93131]]
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     93227
           1       0.98      1.00      0.99     93226

    accuracy                           0.99    186453
   macro avg       0.99      0.99      0.99    186453
weighted avg       0.99      0.99      0.99    186453



In [20]:
from sklearn.metrics import accuracy_score

# Define the threshold (usually 0.5 for binary classification)
threshold = 0.5

# Convert probabilities into predicted class labels based on the threshold
y_pred_probabilities = (class_1_probabilities >= threshold).astype(int)

# Calculate accuracy by comparing predicted labels (y_pred_probabilities) with actual labels (y_test)
accuracy = accuracy_score(y_test, y_pred_probabilities)

print(f"Accuracy based on probabilities: {accuracy}")


Accuracy based on probabilities: 0.9914187489608641


In [21]:
print(classification_report(y_test, y_pred_probabilities))


              precision    recall  f1-score   support

           0       1.00      0.98      0.99     93227
           1       0.98      1.00      0.99     93226

    accuracy                           0.99    186453
   macro avg       0.99      0.99      0.99    186453
weighted avg       0.99      0.99      0.99    186453



In [17]:
import joblib
joblib.dump(model, 'crime_model.pkl')


['crime_model.pkl']

cos_hour,sin_hour,cos_weekday,sin_weekday,Longitude,Latitude,label
1.0,0.0,0.6235,-0.7818,-87.74314,41.80713,1

In [39]:
lat = 41.072
lon = -87.74314
hour = 0        # 8 PM
weekday = 0       # Saturday

import numpy as np

sin_hour = np.sin(2 * np.pi * hour / 24)
cos_hour = np.cos(2 * np.pi * hour / 24)

sin_weekday = np.sin(2 * np.pi * weekday / 7)
cos_weekday = np.cos(2 * np.pi * weekday / 7)

test_point = [[lat, lon, sin_hour, cos_hour, sin_weekday, cos_weekday]]
prediction = model.predict_proba(test_point)
crime_prob = prediction[:,1][0] * 100
print(f"Probability of crime: {crime_prob:.2f}%")


Probability of crime: 13.00%


c:\Users\Administrator\Documents\Repos\CrimeProject\.venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
